# 🤖 Uncertainty Quantification for Large Language Models: White-Box Methods

## 🎯 Motivation
Large language models can generate fluent and convincing answers even when those answers are incorrect, unsupported, or ambiguous. This behaviour makes it difficult to judge whether a response should be trusted based only on its wording.

**Uncertainty Quantification (UQ)** addresses this problem by estimating how reliable a model’s prediction or generated response is. Instead of returning only an answer, the system also produces a confidence or uncertainty score that reflects how strongly the model supports that answer.

UQ is especially important for text generation because an LLM produces a sequence of tokens rather than selecting one label from a fixed set of classes. A response may contain multiple claims, partially correct statements, or several plausible formulations. Therefore, uncertainty must be studied at different levels, including:

- **token level** : uncertainty associated with individual generated tokens
- **sequence level** : uncertainty of the complete response
- **claim level** : uncertainty of specific factual statements
- **multi-generation level** : disagreement across several sampled responses


This tutorial explores practical LLM uncertainty quantification using three open-source libraries:

- **[UQLM](https://github.com/cvs-health/uqlm)**: provides white-box, black-box, LLM-as-a-judge, ensemble, and claim-level uncertainty methods.

- **[LM-Polygraph](https://github.com/IINemo/lm-polygraph)**: provides information-based, attention-based, meaning-diversity, density-based, reflexive, ensemble, and claim-level uncertainty estimators.

- **[LLM Uncertainty Head](https://github.com/IINemo/llm-uncertainty-head)**: provides supervised learned uncertainty using a pretrained uncertainty head attached to a compatible language model.

These libraries are complementary and differ in supported estimators, model requirements, score direction, and computational cost.



## 📑 Tutorial Index

1. [Global Setup](#global-setup)
2. [UQLM White-Box Scorers](#uqlm-whitebox)
3. [UQLM Ensemble Scoring](#uqlm-ensemble)
4. [LM-Polygraph White-Box Uncertainty Quantification](#lm-polygraph-whitebox)
5. [Claim-Level Uncertainty](#claim-level)
6. [Supervised Uncertainty Head](#supervised-uhead)


## ⚪ White-Box Uncertainty Quantification for Large Language Models

This notebook covers the **white-box uncertainty quantification** component of the group project.

White-box methods require access to internal model information produced during generation, such as token probabilities, log-probabilities, hidden representations, and attention scores. These signals are used to estimate how strongly the model supports its own response.

Typical white-box signals include:

* token probability
* sequence probability
* entropy and negentropy
* probability margins
* hidden-state representations
* attention-based uncertainty
* agreement across sampled generations

In general, lower token probabilities, higher entropy, disagreement across generations, or unstable internal representations may indicate greater uncertainty.

White-box methods are often efficient because many can be computed from a single generation. However, they can only be applied when the model or API exposes the required internal information.


The other components of the project, including black-box uncertainty quantification, multimodal uncertainty, normalization, calibration, and benchmarking, are covered by the other team members.


## 🎨 UQ Category Map

| Category | Main idea | Techniques demonstrated |
|---|---|---|
| 🔵 **Information-based** | Uses token probabilities, sequence probabilities, likelihoods, entropy, or probability margins. | Sequence Probability, Minimum Token Probability, Mean Top-K Token Negentropy, Min Top-K Token Negentropy, Probability Margin, Monte Carlo Sequence Probability, Maximum Sequence Probability, Maximum Token Probability, Perplexity, Monte Carlo Sequence Entropy |
| 🟢 **Meaning-diversity** | Measures semantic agreement, disagreement, or variation across generated responses or claims. | CoCoA, Semantic Negentropy, Semantic Density, Semantic Entropy, CoCoA (MSP), CoCoA (PPL), Claim-Conditioned Probability |
| 🟣 **Reflexive** | Asks the model to assess whether its own response or claim is likely to be true. | P(True), P(True) Claim |
| 🟤 **Attention-based** | Uses attention patterns or attention-derived internal signals. | RAUQ, Attention Score |
| 🟠 **Density-based** | Compares internal model representations with a fitted in-domain reference distribution. | Robust Density Estimation |
| ⚫ **Ensemble** | Combines several complementary confidence or uncertainty signals into one score. | UQLM Off-the-Shelf Ensemble |
| 🔴 **Learned uncertainty** | Uses a supervised uncertainty head trained on internal model representations. | Supervised Uncertainty Head |


## 🧭 Score-Direction Awareness

Different libraries may report either confidence or uncertainty:

* **Confidence:** a higher value indicates that the response is considered more reliable.
* **Uncertainty:** a higher value indicates that the response is considered less reliable.

UQLM mainly reports confidence scores in the `[0,1]` range, while LM-Polygraph generally reports uncertainty on estimator-specific scales. The score direction should therefore always be checked before interpreting or comparing results.

> **Method-classification note:** Some multi-generation consistency methods can also be used in black-box settings because they rely primarily on generated responses. They are included here because they are evaluated through the white-box model and library pipeline used in this notebook.


## 🧰 Why this notebook uses `uq_toolbox`

`uq_toolbox` is not an external UQ library or a new uncertainty method. It is a small project utility created to reduce repeated setup code and keep the notebook concise.

It handles shared tasks such as:

- loading and registering the required models;
- assigning simple model aliases;
- reusing compatible model interfaces across UQLM and LM-Polygraph;
- storing common configuration;
- cleaning up models before loading the supervised uncertainty head.

The actual uncertainty estimators still come from **UQLM**, **LM-Polygraph**, and the **pretrained LLM uncertainty head**. The toolbox only removes infrastructure boilerplate so the notebook can focus on the methods being taught.

| Module | Role |
|---|---|
| `model_manager.py` | Loads and registers model interfaces |
| `density_uq.py` | Runs the LM-Polygraph RDE workflow |
| `claim_uq.py` | Runs claim extraction and claim-level scoring |
| `learned_uq/` | Loads and applies the supervised uncertainty head |

<a id="global-setup"></a>

## ⚙️ Global Setup

The global setup is executed once at the beginning of the notebook. It prepares the shared environment used throughout the tutorial by:

1. installing the required libraries;
2. loading the `uq_toolbox` project utility;
3. collecting the required credentials;
4. defining the medical prompts;
5. initialising the UQLM and LM-Polygraph models.

All later sections reuse these prompts and model instances. A separate model is loaded only for the supervised uncertainty-head section because the pretrained head requires a compatible Mistral backbone.

> **Note:** Run the following setup cells once before executing the rest of the notebook.


### ✅ Requirements

Before running the notebook, make sure the following resources are available:

- **OpenAI API key** : required for the UQLM white-box methods and for claim extraction in the LM-Polygraph claim-level section.
- **Hugging Face access token** : required to download the local language models, auxiliary models, and pretrained uncertainty head from Hugging Face.
- **GPU runtime** : a CUDA-enabled GPU is strongly recommended for the LM-Polygraph and supervised uncertainty sections.
- **Recommended GPU** : an NVIDIA T4 or better should be sufficient for most of the notebook. The supervised uncertainty-head section is the most memory-intensive because it loads Mistral-7B together with its compatible uncertainty head.
- **Internet connection** : required during setup to install the libraries and download model checkpoints.

> **Colab users:** select **Runtime → Change runtime type → T4 GPU** before running the notebook.

> **⚠️ Hardware requirement:** The Supervised Uncertainty Head sections load Mistral-7B together with its compatible uncertainty head and therefore requires substantially more GPU memory than the previous sections. A high-memory GPU is recommended.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
# Core UQ, model, data, and visualisation packages.
%pip install -q "pip<24.1"
%pip install -q -U \
  uqlm langchain-openai pandas plotly matplotlib \
  transformers accelerate sentence-transformers datasets \
  scikit-learn nltk ipywidgets
%pip install -q \
  "git+https://github.com/IINemo/lm-polygraph.git@dev" \
  "git+https://github.com/IINemo/llm-uncertainty-head.git"
%pip install -q --force-reinstall "protobuf==5.28.3"
%pip install -q langchain-anthropic
%pip install -q langchain-google-genai
%pip install -qU langchain-ollama

> ⚠️ **Mandatory Runtime Restart**
>
> After running the cell above (installations), you **must restart the Colab session** before continuing.
>
> Go to **Runtime → Restart session**, then run all cells below this point in order. Skipping this step may cause dependency conflicts or runtime errors.

In [1]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

print("All provider packages imported successfully.")

All provider packages imported successfully.


In [2]:
import os

# Force Transformers to use PyTorch and suppress tokenizer warnings.
os.environ.update({
    "USE_TF": "0",
    "TRANSFORMERS_NO_TF": "1",
    "USE_TORCH": "1",
    "TOKENIZERS_PARALLELISM": "false",
})

In [3]:
import subprocess
import sys
from pathlib import Path

repo_dir = Path("/content/Whitebox-UQ")

subprocess.run(
    ["rm", "-rf", str(repo_dir)],
    check=True,
)

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/zzunairaa/Whitebox-UQ.git",
        str(repo_dir),
    ],
    check=True,
)

sys.path.insert(0, str(repo_dir))

from uq_toolbox import initialize_uq_models
from uq_toolbox.core.claim_uq import run_claim_level_uq
from uq_toolbox.learned_uq import (
    SupervisedUQManager,
    evaluate_supervised_batch,
)

print("All toolbox imports succeeded.")

All toolbox imports succeeded.


In [4]:
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
os.environ.pop("OPENAI_BASE_URL", None)

Enter your OpenAI API key: ··········


In [5]:
import random
import numpy as np
import pandas as pd
import plotly.express as px
import torch

from IPython.display import display
from uqlm import WhiteBoxUQ, UQEnsemble
from lm_polygraph import estimate_uncertainty

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 110)

### 🏥 Medical-Domain Prompts

The same two medical-domain prompts are reused throughout the notebook so that differences in the results reflect the uncertainty method or model rather than changes in the input.


In [6]:
prompt_labels = [
    "Prompt 1 — Anaphylaxis",
    "Prompt 2 — Chest pain",
]

prompts = [
    "A 60-year-old patient with a known penicillin allergy is mistakenly given amoxicillin and develops swelling of the lips and tongue within minutes. What is the most appropriate immediate treatment?",
    "A 50-year-old man presents with crushing chest pain radiating to his left arm, sweating, and shortness of breath for the past 30 minutes. What is the immediate first-line management?",
]

for label, prompt in zip(prompt_labels, prompts):
    print(f"{label}: {prompt}\n")

Prompt 1 — Anaphylaxis: A 60-year-old patient with a known penicillin allergy is mistakenly given amoxicillin and develops swelling of the lips and tongue within minutes. What is the most appropriate immediate treatment?

Prompt 2 — Chest pain: A 50-year-old man presents with crushing chest pain radiating to his left arm, sweating, and shortness of breath for the past 30 minutes. What is the immediate first-line management?



In [7]:
uq_context = initialize_uq_models(
    polygraph_models={
        "qwen": {
            "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
            "mode": "white",
            "device_map": "cuda:0",
            "torch_dtype": torch.float32,
            "attn_implementation": "eager",
            "trust_remote_code": True,
        }
    },
    uqlm_models={
        "openai": {
            "provider": "openai",
            "model_id": "gpt-4o",
            "mode": "white",
            "temperature": 0.2,
            "max_tokens": 180,
            "kwargs": {"top_logprobs": 20},
        }
    },
    p_mode="white",
    u_mode="white",
    temperature=0.2,
    max_tokens=180,
)

uq_context.list_active_models()


INITIALISING UQ MODEL ENVIRONMENT

Adding LM-Polygraph [qwen] in WHITE-BOX mode...
Paste your Hugging Face token: ··········


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Adding UQLM [openai] in WHITE-BOX mode...


/content/Whitebox-UQ/uq_toolbox/managers/model_manager.py:892: UserWarning: Parameters {'logprobs'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  manager.add_uqlm_model(



UQ MODEL ENVIRONMENT READY

LIVE UNCERTAINTY QUANTIFICATION REGISTRIES
LM-Polygraph framework (1 active):
  Alias: qwen — Operational mode: WHITE-BOX

UQLM framework (1 active):
  Alias: openai — Operational mode: WHITE-BOX



In [8]:
# UQLM uses the registered LangChain model.
llm = uq_context.langchain_llms["openai"]

# LM-Polygraph uses the registered local white-box model.
model = uq_context.polygraph_models["qwen"]
base_model = model.model
tokenizer = model.tokenizer

print("UQLM model: openai")
print("LM-Polygraph model: qwen")

UQLM model: openai
LM-Polygraph model: qwen


### 📚 UQLM Library Overview

[UQLM](https://github.com/cvs-health/uqlm) is a library for estimating the confidence of large language model responses. It supports white-box, black-box, LLM-judge, and ensemble-based scorers through a unified interface.

The general workflow is:

1. provide one or more prompts;
2. generate a primary response and, when required, additional candidate responses;
3. compute one or more uncertainty or confidence signals;
4. return the generated text together with response-level scores.

<div style="text-align:center; margin:24px 0;">
  <img
    src="https://raw.githubusercontent.com/cvs-health/uqlm/main/assets/images/uqlm_flow_ds.png"
    alt="UQLM library workflow"
    style="width:100%; max-width:1100px; height:auto;"
  >
  <p style="font-size:14px; color:#666; font-style:italic;">
    Figure 1. UQLM workflow from prompt generation to confidence scoring.
  </p>
</div>

In this notebook, UQLM is used for individual white-box scorers and the off-the-shelf ensemble. Its scores are generally reported in `[0,1]`, where higher values indicate greater confidence.

<a id="uqlm-whitebox"></a>

# 1️⃣ UQLM White-Box Scorers

White-box scorers use the probabilities assigned to generated tokens to estimate uncertainty. They can often score a response from a single generation, making them significantly faster and cheaper than methods that require several sampled answers.

However, they require access to the model's internal token probabilities or log-probabilities, so they are not compatible with every language model or API.

[UQLM](https://github.com/cvs-health/uqlm) converts these signals into response-level **confidence scores** in `[0,1]`, where higher values indicate greater confidence.

| Group | Scorers demonstrated | Additional generation cost |
|---|---|---|
| ⚡ **Single generation** | Sequence Probability, Minimum Token Probability, Mean Token Negentropy, Minimum Token Negentropy, Probability Margin | None |
| 🪞 **Self-reflection** | P(True) | One extra generation |
| 🔁 **Multiple generations** | Monte Carlo Sequence Probability, CoCoA, Semantic Negentropy, Semantic Density | Several generations |

Single-generation scorers reuse one response and its token probabilities. Multi-generation scorers additionally sample several responses to combine probability information with semantic agreement.

> **Interpretation:** UQLM reports confidence, so higher scores indicate greater confidence and lower scores indicate greater uncertainty.

<div style="text-align:center; margin:24px 0;">
  <img
    src="https://raw.githubusercontent.com/cvs-health/uqlm/main/assets/images/white_box_graphic.png"
    alt="UQLM white-box scoring workflow"
    style="width:100%; max-width:1100px; height:auto;"
  >
  <p style="font-size:14px; color:#666; font-style:italic;">
    Figure 1. White-box scorers use token-level probabilities to produce a response-level confidence score.
  </p>
</div>

---

#### 🔬 How UQLM Works Natively

The native UQLM workflow has three steps: create a LangChain-compatible language model, initialise the required scorer, and call it on one or more prompts.

```python
from langchain_openai import ChatOpenAI
from uqlm import WhiteBoxUQ

llm = ChatOpenAI(model="gpt-4o")

scorer = WhiteBoxUQ(llm=llm, scorers=["sequence_probability"])

result = await scorer.generate_and_score(prompts=prompts)
```



For the complete native API and supported scorers, see the official [UQLM documentation and examples](https://github.com/cvs-health/uqlm).

---

#### 🧰 Model Access via `uq_toolbox`

The notebook uses UQLM's native scorer classes directly. `uq_toolbox` is used only to load and register the LangChain-compatible model once so it can be reused across the tutorial:

```python
llm = uq_context.langchain_llms["openai"]
```

The native UQLM scorer is then initialised normally:

```python
scorer = WhiteBoxUQ(llm=llm, scorers=["sequence_probability"])
result = await scorer.generate_and_score(prompts=prompts)
```

`uq_toolbox` does not wrap or replace the UQLM scoring API. It only centralises model configuration and avoids repeating authentication and model-loading code.

## 🧰 Experiment Helper

The helper runs one UQLM scorer on the shared prompts, displays the relevant outputs, and stores the results for later comparison. Model setup remains handled by `uq_toolbox`.

In [9]:
results_store = {}

async def run_scorer(
    scorer,
    num_responses=None,
    sampling_temperature=None,
    display_columns=None,
):
    options = {"llm": llm, "scorers": [scorer]}
    if sampling_temperature is not None:
        options["sampling_temperature"] = sampling_temperature

    run_options = {"prompts": prompts}
    if num_responses is not None:
        run_options["num_responses"] = num_responses

    result = await WhiteBoxUQ(**options).generate_and_score(**run_options)
    df = result.to_df().copy()
    df.insert(0, "prompt_label", prompt_labels)
    results_store[scorer] = df

    columns = display_columns or ["prompt_label", "response", "logprob", scorer]
    display(df[[c for c in columns if c in df.columns]])

    return df

## ⚡ Single-Generation Scorers (minimal latency, zero extra cost)

### ⚡ 1. Sequence Probability

**What it measures:** confidence in the complete response from its generated-token probabilities.

**What it requires:** one model generation with access to the log-probability of each generated token.

**How to interpret the score:** higher values indicate greater confidence in the generated sequence.

For a detailed explanation, see [Vashurin et al. (2025)](https://arxiv.org/abs/2406.15627).


In [10]:
sequence_df = await run_scorer("sequence_probability")

Output()

,prompt_label,response,logprob,sequence_probability
0,Prompt 1 — Anaphylaxis,"The symptoms described are indicative of an anaphylactic reaction, which is a severe and potentially life-...","[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.2867041528224945, 'top_logprobs': [{'token': 'The...",0.817221
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.4461166262626648, 'top_logprobs': [{'token': 'The...",0.736339


**Output:** The model returns token-level `logprob` values for the generated response. UQLM aggregates these token probabilities into one `sequence_probability` confidence score for the complete sequence.

### ⚡ 2. Minimum Token Probability

**What it measures:** response confidence using the probability of the least likely generated token.

**What it requires:** one model generation with access to token-level probabilities or log-probabilities.

**How to interpret the score:** lower values indicate that at least one token was generated with low confidence.

For more detail, see [Manakul et al. (2023)](https://arxiv.org/abs/2303.08896).


In [11]:
minimum_probability_df = await run_scorer("min_probability")

Output()

,prompt_label,response,logprob,min_probability
0,Prompt 1 — Anaphylaxis,The symptoms described suggest that the patient is experiencing an anaphylactic reaction due to the amoxic...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.2867041528224945, 'top_logprobs': [{'token': 'The...",0.224058
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.44057372212409973, 'top_logprobs': [{'token': 'Th...",0.192811


**Output:** UQLM checks the probability of every generated token and returns the smallest one as `min_probability`. A low value means at least one token in the response was generated with weak confidence.

### ⚡ 3. Mean Top-K Token Negentropy

**What it measures:** how concentrated the model's Top-\(K\) next-token distribution is, averaged across generated positions.

**What it requires:** one model generation with access to the Top-\(K\) next-token probability distribution at each generation step.

**How to interpret the score:** higher values indicate a more concentrated distribution and therefore greater confidence.

For more detail, see [Scalena et al. (2025)](https://arxiv.org/abs/2510.11170) and [Manakul et al. (2023)](https://arxiv.org/abs/2303.08896).


In [12]:
mean_negentropy_df = await run_scorer("mean_token_negentropy")

Output()

/usr/local/lib/python3.12/dist-packages/uqlm/scorers/shortform/white_box.py:227: UQLMBetaWarning: Scorers based on top_logprobs ('mean_token_negentropy','min_token_negentropy','probability_margin') is in beta. Please use with caution as it may change in future releases.
  beta_warning("Scorers based on top_logprobs ('mean_token_negentropy','min_token_negentropy','probability_margin') is in beta. Please use with caution as it may change in future releases.")


,prompt_label,response,logprob,mean_token_negentropy
0,Prompt 1 — Anaphylaxis,The symptoms described indicate that the patient is experiencing an acute anaphylactic reaction due to the...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.27220967411994934, 'top_logprobs': [{'token': 'Th...",0.850564
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.48993438482284546, 'top_logprobs': [{'token': 'Th...",0.816914


**Output:** UQLM computes Top-\(K\) token negentropy at each generation step and averages it across the response. Higher `mean_token_negentropy` means the model’s next-token distributions were generally more concentrated and confident.

### ⚡ 4. Min Top-K Token Negentropy

**What it measures:** the generated position with the least concentrated Top-\(K\) next-token distribution.

**What it requires:** one model generation with access to the Top-\(K\) next-token probability distribution at each generation step.

**How to interpret the score:** lower values indicate a weaker confidence point in the response.

For more detail, see [Scalena et al. (2025)](https://arxiv.org/abs/2510.11170) and [Manakul et al. (2023)](https://arxiv.org/abs/2303.08896).


In [13]:
minimum_negentropy_df = await run_scorer("min_token_negentropy")


Output()

/usr/local/lib/python3.12/dist-packages/uqlm/scorers/shortform/white_box.py:227: UQLMBetaWarning: Scorers based on top_logprobs ('mean_token_negentropy','min_token_negentropy','probability_margin') is in beta. Please use with caution as it may change in future releases.
  beta_warning("Scorers based on top_logprobs ('mean_token_negentropy','min_token_negentropy','probability_margin') is in beta. Please use with caution as it may change in future releases.")


,prompt_label,response,logprob,min_token_negentropy
0,Prompt 1 — Anaphylaxis,"The symptoms described are indicative of an anaphylactic reaction, which is a severe and potentially life-...","[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.27894583344459534, 'top_logprobs': [{'token': 'Th...",0.162466
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.4461166262626648, 'top_logprobs': [{'token': 'The...",0.219789


**Output:** UQLM computes Top-\(K\) token negentropy at every generation step and returns the minimum value. This identifies the least confident token position in the response.

### ⚡ 5. Probability Margin

**What it measures:** the gap between the most likely and second-most likely next-token choices.

**What it requires:** one model generation with access to the highest-ranked next-token probabilities at each generation step.

**How to interpret the score:** higher values indicate a clearer and more confident token choice.

For more detail, see [Farr et al. (2024)](https://arxiv.org/abs/2408.08217)


In [14]:
probability_margin_df = await run_scorer("probability_margin")

Output()

/usr/local/lib/python3.12/dist-packages/uqlm/scorers/shortform/white_box.py:227: UQLMBetaWarning: Scorers based on top_logprobs ('mean_token_negentropy','min_token_negentropy','probability_margin') is in beta. Please use with caution as it may change in future releases.
  beta_warning("Scorers based on top_logprobs ('mean_token_negentropy','min_token_negentropy','probability_margin') is in beta. Please use with caution as it may change in future releases.")


,prompt_label,response,logprob,probability_margin
0,Prompt 1 — Anaphylaxis,The symptoms described suggest that the patient is experiencing an acute anaphylactic reaction due to the ...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.27220967411994934, 'top_logprobs': [{'token': 'Th...",0.773807
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.4461166262626648, 'top_logprobs': [{'token': 'The...",0.718482


**Output:** UQLM computes the probability gap between the top two token candidates at each generation step, then summarizes these margins into one `probability_margin` confidence score. A larger value means the model usually had a clearer preferred token.

## 🪞 Self-Reflection Scorer (one additional generation per response)

### 🪞 P(True)

**What it measures:** the model's self-assessed probability that its previously generated answer is true.

**What it requires:** the original generated response and one additional self-evaluation call in which the model judges whether the answer is `True` or `False`.

**How to interpret the score:** higher values indicate greater self-assessed confidence in the answer.

For more detail, see [Kadavath et al. (2022)](https://arxiv.org/abs/2207.05221).


In [15]:
p_true_df = await run_scorer(
    "p_true",
)

Output()

,prompt_label,response,logprob,p_true
0,Prompt 1 — Anaphylaxis,The symptoms described indicate that the patient is experiencing an acute anaphylactic reaction due to the...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.23746071755886078, 'top_logprobs': [{'token': 'Th...",0.999997
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.4461166262626648, 'top_logprobs': [{'token': 'The...",0.949668


**Output:** UQLM performs an additional self-evaluation step and returns the probability of `True` as `p_true`. A value close to `1` means the model strongly judges its own response as correct.

## 🔁 Multi-Generation Scorers (several additional generations per response)


Each method generates **5 candidate responses per prompt** using a **sampling temperature of `1.0`**. The higher temperature introduces response variation needed to measure probability, consistency, and semantic agreement.

### 🔁 1. Monte Carlo Sequence Probability

**What it measures:** average confidence across several sampled responses using their length-normalised sequence probabilities.

**What it requires:** multiple sampled responses with access to their token log-probabilities.

**How to interpret the score:** higher values indicate that the sampled responses are collectively more probable under the model.

For more detail, see [Kuhn et al. (2023)](https://arxiv.org/abs/2302.09664).


In [16]:
monte_carlo_df = await run_scorer(
    "monte_carlo_probability",
    num_responses=5,
    sampling_temperature=1.0,
)

Output()

,prompt_label,response,logprob,monte_carlo_probability
0,Prompt 1 — Anaphylaxis,The symptoms described indicate that the patient is experiencing an acute anaphylactic reaction due to the...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.363134503364563, 'top_logprobs': [{'token': 'The'...",0.592112
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.4461166262626648, 'top_logprobs': [{'token': 'The...",0.327496


**Output:** UQLM generates five candidate responses per prompt, computes their length-normalized sequence probabilities, and averages them into one `monte_carlo_probability` confidence score.

### 🔁  2. Consistency and Confidence (CoCoA)

**What it measures:** response confidence by combining sequence probability with semantic agreement across sampled responses.

**What it requires:** multiple sampled responses, sequence-probability information, and a semantic-similarity model for comparing the generations.

**How to interpret the score:** higher values indicate that the response is both probable and consistent with alternative generations.

For more detail, see [Vashurin et al. (2025)](https://arxiv.org/abs/2502.04964).


In [17]:
cocoa_df = await run_scorer(
    "consistency_and_confidence",
    num_responses=5,
    sampling_temperature=1.0,
)

Output()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,prompt_label,response,logprob,consistency_and_confidence
0,Prompt 1 — Anaphylaxis,"The symptoms described are indicative of an anaphylactic reaction, which is a severe and potentially life-...","[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.27220967411994934, 'top_logprobs': [{'token': 'Th...",0.769635
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.4461166262626648, 'top_logprobs': [{'token': 'The...",0.746505


**Output:** UQLM scores the original response using its token probabilities and compares it with sampled responses using cosine similarity. These signals are combined into one `consistency_and_confidence` score.

### 🔁  3. Semantic Negentropy

**What it measures:** how concentrated the sampled responses are across semantic meaning clusters.

**What it requires:** multiple sampled responses and a semantic comparison method for grouping responses that express the same meaning.

**How to interpret the score:** higher values indicate that the samples support fewer competing meanings.

For more detail, see [Farquhar et al. (2024)](https://arxiv.org/abs/2406.15927).


In [18]:
semantic_negentropy_df = await run_scorer(
    "semantic_negentropy",
    num_responses=5,
    sampling_temperature=1.0,
)

Output()

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

,prompt_label,response,logprob,semantic_negentropy
0,Prompt 1 — Anaphylaxis,The symptoms described indicate that the patient is experiencing an acute anaphylactic reaction due to the...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.363134503364563, 'top_logprobs': [{'token': 'The'...",1.0
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.4461166262626648, 'top_logprobs': [{'token': 'The...",1.0


**Output:** UQLM generates five candidate responses per prompt, groups them into semantic clusters, and converts the resulting semantic entropy into `semantic_negentropy`. A score of `1.0` means the sampled responses were concentrated in the same semantic interpretation.

### 🔁 4. Semantic Density

**What it measures:** how densely the primary response is supported by semantically similar sampled answers.

**What it requires:** multiple sampled responses and a semantic embedding or similarity model for comparing their meanings.

**How to interpret the score:** higher values indicate stronger support in semantic space.

For more detail, see [Qiu and Miikkulainen (2024)](https://arxiv.org/abs/2405.13845).


In [19]:
semantic_density_df = await run_scorer(
    "semantic_density",
    num_responses=5,
    sampling_temperature=1.0,
)

Output()

,prompt_label,response,logprob,semantic_density
0,Prompt 1 — Anaphylaxis,"The symptoms described are indicative of an anaphylactic reaction, which is a severe and potentially life-...","[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.2867041528224945, 'top_logprobs': [{'token': 'The...",0.985836
1,Prompt 2 — Chest pain,The symptoms described are highly suggestive of an acute myocardial infarction (heart attack). The immedia...,"[{'token': 'The', 'bytes': [84, 104, 101], 'logprob': -0.4778488278388977, 'top_logprobs': [{'token': 'The...",0.930998


**Output:** UQLM generates five candidate responses, groups them by semantic similarity, and computes how densely the original response is supported by those samples. Higher `semantic_density` indicates stronger semantic support.

## 📊 UQLM White-Box Summary

The graph compares all UQLM white-box scorers across the two medical prompts. Each method captures a different aspect of uncertainty, including sequence probability, token confidence, entropy, semantic consistency and multi-generation agreement.

In [20]:
import pandas as pd
import plotly.express as px

uqlm_scores_df = pd.concat(
    [
        df[["prompt_label", scorer]]
        .rename(columns={scorer: "confidence"})
        .assign(scorer=scorer)
        for scorer, df in results_store.items()
        if scorer in df.columns
    ],
    ignore_index=True,
)

px.bar(
    uqlm_scores_df,
    x="scorer",
    y="confidence",
    color="prompt_label",
    barmode="group",
    title="UQLM White-Box Confidence Scores",
    labels={
        "scorer": "Scorer",
        "confidence": "Confidence score",
        "prompt_label": "Prompt",
    },
).show()

## 📏 Interpreting UQLM Confidence Scores

UQLM returns all confidence scores on a common **[0, 1]** range:

- Scores closer to **1** indicate higher confidence.
- Scores closer to **0** indicate lower confidence and greater uncertainty.

This shared scale makes individual scores easy to read. However, placing different scorers on the same range does not make them directly comparable ,each method measures a different signal, such as token probability, entropy, semantic agreement, or self-evaluation.

A score of `0.8` from one method does not necessarily represent the same level of reliability as `0.8` from another.

> **Note:** Formal cross-estimator comparison requires normalisation or calibration against labelled development data.

<a id="uqlm-ensemble"></a>

---
# 2️⃣ UQLM Ensemble Scoring

A single uncertainty scorer may capture only one aspect of model confidence. UQLM ensemble scorers combine complementary confidence signals through a weighted average to produce one final `ensemble_score`.

The off-the-shelf ensemble used here includes:

- `noncontradiction`, which measures whether sampled responses support rather than contradict the primary response;
- `exact_match`, which measures exact textual agreement across generated responses;
- `judge_1`, which uses an LLM judge to assess the response;
- `ensemble_score`, which combines the component scores using predefined weights.

<div style="text-align:center; margin:24px 0;">
  <img
    src="https://raw.githubusercontent.com/cvs-health/uqlm/main/assets/images/uqensemble_generate_score.png"
    alt="UQLM ensemble generation and scoring workflow"
    style="width:100%; max-width:1100px; height:auto;"
  >
  <p style="font-size:14px; color:#666; font-style:italic;">
    Figure 2. UQLM generates responses, computes several confidence signals, and combines them into an ensemble score.
  </p>
</div>



## Ensemble modes in UQLM

UQLM supports two ensemble configurations.

### 1️. Off-the-shelf ensemble


The off-the-shelf ensemble uses predefined scorers and fixed weights. It can be applied directly without labelled examples, calibration data, or an additional training step.

This configuration is based on the **BSDetector** approach introduced by [Chen and Mueller (2023)](https://arxiv.org/abs/2308.16175).

### 2️. Supervised ensemble

The supervised ensemble uses `UQEnsemble.tune(...)` to learn the contribution of selected scorers from prompts with known ground-truth answers. This allows the ensemble to adapt its weights to a particular task or domain.

For more detail, see [Bouchard and Chauhan (2025)](https://arxiv.org/abs/2504.19254).

## Ensemble variant used in this notebook

This notebook uses the **off-the-shelf UQLM ensemble**.

The supervised ensemble is not used because the notebook contains only a small number of demonstration prompts and does not include a separate labelled tuning dataset. Supervised tuning requires enough prompts with reliable ground-truth answers to learn meaningful scorer weights.

Applying `UQEnsemble.tune(...)` to only a few tutorial examples could produce unreliable weights. The off-the-shelf ensemble is therefore more appropriate for demonstrating ensemble scoring directly.

**What it measures:** overall response confidence by combining complementary probability-, consistency-, and self-evaluation signals.

**What it requires:** a primary response, multiple sampled alternatives, and the component scorers included in the predefined ensemble.

**How to interpret the score:** a higher `ensemble_score` indicates greater combined confidence. The component scores can be inspected to understand which signals contributed to the result.

## 🚀 Run the Off-the-Shelf Ensemble

The ensemble performs three main steps:

1. generates a primary response;
2. generates five alternative responses for consistency-based scoring;
3. combines the component confidence scores into one `ensemble_score`.

The component scores are retained in the output, making it possible to inspect why the final ensemble confidence is high or low.

In [21]:
# Keep generated medical responses concise and easier to compare.
CONCISE_SYSTEM_PROMPT = (
    "You are a helpful medical assistant. "
    "Answer clearly and concisely, in under 150 words unless more detail is necessary."
)

# Reuse the OpenAI model already registered by uq_toolbox.
ensemble_llm = uq_context.langchain_llms["openai"]

uqe = UQEnsemble(
    llm=ensemble_llm,
    system_prompt=CONCISE_SYSTEM_PROMPT,
    max_length=1024,
)

ensemble_result = await uqe.generate_and_score(
    prompts=prompts,
    num_responses=5,
)

ensemble_df = ensemble_result.to_df().copy()
ensemble_df.insert(0, "prompt_label", prompt_labels)

display(
    ensemble_df[
        [
            "prompt_label",
            "response",
            "sampled_responses",
            "noncontradiction",
            "exact_match",
            "judge_1",
            "ensemble_score",
        ]
    ]
)

Output()

,prompt_label,response,sampled_responses,noncontradiction,exact_match,judge_1,ensemble_score
0,Prompt 1 — Anaphylaxis,"The most appropriate immediate treatment for this patient, who is experiencing signs of anaphylaxis due to...","[The most appropriate immediate treatment for this patient, showing signs of an allergic reaction such as ...",0.997013,0.0,1.0,0.858327
1,Prompt 2 — Chest pain,The immediate first-line management for a 50-year-old man presenting with symptoms suggestive of an acute ...,[The immediate first-line management for a 50-year-old man presenting with symptoms suggestive of acute my...,0.995057,0.0,1.0,0.857232


**Output:** UQLM returns the primary response, five sampled alternatives, the individual component confidence scores, and the final `ensemble_score`. Higher values indicate greater confidence.

Because `exact_match` requires identical text, it may be low even when sampled responses express the same medical meaning using different wording. In such cases, `noncontradiction` provides a more semantic measure of agreement.

---


## 📚 LM-Polygraph Library

[LM-Polygraph](https://github.com/IINemo/lm-polygraph) is an open-source Python library for estimating uncertainty in language-model generations.

It provides a unified interface for a broad range of uncertainty-estimation methods, including:

- 🔵 information-based methods;
- 🟢 semantic and meaning-diversity methods;
- 🟤 attention-based methods;
- 🟠 density-based methods;
- 🟣 reflexive methods;

The library supports both white-box and black-box techniques. White-box methods use internal model information such as token probabilities, logits, attention values, or hidden representations, while black-box methods rely only on generated text.


<a id="lm-polygraph-whitebox"></a>

---

# 3️⃣ LM-Polygraph White-Box Uncertainty Quantification

LM-Polygraph white-box methods estimate uncertainty using information available inside the language model, including token probabilities and logits, attention values, hidden representations, and probabilities across multiple sampled generations.

Because these methods require access to model internals, they are generally used with open or locally loaded models rather than closed APIs that expose only generated text.

LM-Polygraph supports several levels of granularity:

- **Sequence level:** one uncertainty score for the complete response.
- **Token level:** one uncertainty score for each generated token.
- **Claim level:** one uncertainty score for each factual claim.

> **Interpretation:** LM-Polygraph generally reports uncertainty, so higher values usually indicate greater uncertainty. Because estimators may use different numerical scales, compare the same estimator across prompts rather than comparing unrelated methods directly.

> **Backbone choice:** This tutorial uses `Qwen/Qwen2.5-0.5B-Instruct` as a lightweight and Colab-friendly backbone. It exposes the model internals required by the demonstrated estimators while keeping memory use and execution time manageable.

For further details, see [Fadeeva et al. (2023)](https://aclanthology.org/2023.emnlp-demo.41/) and the official [LM-Polygraph repository](https://github.com/IINemo/lm-polygraph).

---

#### 🔬 How LM-Polygraph Works Natively

A typical sequence-level call requires three steps: load a white-box model, select an estimator, and call `estimate_uncertainty`.

```python
from lm_polygraph.estimators import MaximumSequenceProbability
from lm_polygraph import estimate_uncertainty

result = estimate_uncertainty(
    model=model,
    estimator=MaximumSequenceProbability(),
    input_text=prompt,
)

result.generation_text   # the generated response
result.uncertainty       # the uncertainty score
```

Any estimator class can be swapped in — the workflow is identical regardless of whether the method uses probabilities, attention, hidden representations, or semantic variation.

For the complete native API, see the official [LM-Polygraph documentation](https://github.com/IINemo/lm-polygraph).

---

#### 🧰 Model Access via `uq_toolbox`

The notebook calls LM-Polygraph estimators directly. `uq_toolbox` only loads and registers the Qwen model once so it can be reused throughout the tutorial:

```python
model = uq_context.polygraph_models["qwen"]
```

It does not replace the LM-Polygraph API or modify its estimators. More specialised helpers are used only for workflows that require substantial setup, such as Robust Density Estimation and claim-level uncertainty.

In [22]:
model = uq_context.polygraph_models["qwen"]
tokenizer = model.tokenizer

print("LM-Polygraph alias: qwen")
print("Model wrapper:", type(model).__name__)
print("Tokenizer:", type(tokenizer).__name__)

LM-Polygraph alias: qwen
Model wrapper: WhiteboxModel
Tokenizer: Qwen2TokenizerFast


## 💬 Prepare the prompts for Qwen

Qwen expects instruction-style chat formatting. The original medical questions are unchanged; `apply_chat_template(...)` only adds the model-specific conversation tokens required for generation.

In [23]:
polygraph_prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for prompt in prompts
]

print(f"Prepared {len(polygraph_prompts)} Qwen-formatted prompts.")

Prepared 2 Qwen-formatted prompts.


## 📦 Import the Estimators

The next cell imports the sequence- and token-level estimators used in this section. Claim-level estimators are imported later in their dedicated section.

The helper runs a **sequence-level estimator** on both medical prompts and returns the generated response with one uncertainty score per prompt.

In [24]:
def run_polygraph(estimator):
    rows = []

    for label, prompt in zip(prompt_labels, polygraph_prompts):
        output = estimate_uncertainty(
            model,
            estimator,
            input_text=prompt,
        )

        rows.append({
            "prompt": label,
            "uncertainty": float(output.uncertainty),
            "response": output.generation_text,
        })

    return pd.DataFrame(rows)

In [25]:
from lm_polygraph import estimate_uncertainty

from lm_polygraph.estimators import (
    # Information-based
    MaximumSequenceProbability,
    MaximumTokenProbability,
    Perplexity,
    MonteCarloSequenceEntropy,

    # Attention-based
    RAUQ,
    AttentionScore,

    # Meaning-diversity
    SemanticEntropy,
    CocoaMSP,
    CocoaPPL,

    # Reflexive
    PTrue,
)

> **Scope:** The technique categories in this section are demonstrated only at the **token** and **sequence** levels. Claim-level uncertainty is covered separately later in the notebook.

## 🔵 Information-Based Methods

Information-based estimators use token or sequence probabilities produced by the language model.

These methods are usually efficient because several of them require only one generated response. In LM-Polygraph, they return uncertainty scores rather than normalized confidence scores.

> **Interpretation:** higher values generally indicate greater uncertainty. Compare the same estimator across prompts rather than comparing raw values from unrelated methods.


### 🔵 1. Maximum Sequence Probability

**What it measures:** sequence-level uncertainty from the probability assigned to the complete generated response.

**What it requires:** one model generation with access to token log-probabilities for the complete generated sequence.

**How to interpret the score:** a larger value means the sequence received lower probability and is therefore considered more uncertain.


In [26]:
msp_df = run_polygraph(MaximumSequenceProbability())
display(msp_df)

,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,58.233494,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,90.879959,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


### 🔵 2. Maximum Token Probability

**What it measures:** token-level uncertainty from the probability assigned to each generated token.

**What it requires:** one model generation with access to token-level probability information.

**How to interpret the score:** larger token-level values indicate positions where the model was less certain about its next-token choice.

For more detail, see [Fomicheva et al. (2020)](https://aclanthology.org/2020.tacl-1.35/).


In [27]:
token_results = []
estimator = MaximumTokenProbability()

for label, prompt in zip(prompt_labels, prompts):
    ue = estimate_uncertainty(
        model,
        estimator,
        input_text=prompt,
    )

    token_results.append({
        "prompt": label,
        "uncertainty": ue.uncertainty,
        "response": ue.generation_text,
    })

token_df = pd.DataFrame(token_results)
display(token_df)

,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,"[0.9381701, 0.123001784, 0.16207314, 2.0635748, 0.3130964, 0.56455696, 0.0036093346, 1.8283744, 1.5056047,...",\nA. Intravenous injection of 10% calcium gluconate\nB. Intravenous injection of 5% sodium bicarbonate\nC...
1,Prompt 2 — Chest pain,"[0.9339154, 0.09010348, 2.7725215, 0.13162161, 2.7355945, 0.005695662, 2.41479, 1.4606397, 0.6648719, 0.00...",A: Intravenous administration of a beta-blocker B: Intravenous administration of a calcium channel blocke...


### 🔵 3. Perplexity

**What it measures:** how unlikely the generated response is under the model using length-normalised token probabilities.

**What it requires:** one model generation with access to the log-probability of every generated token.

**How to interpret the score:** higher perplexity indicates that the generated sequence was less expected by the model and is therefore more uncertain.

For more detail, see [Fomicheva et al. (2020)](https://aclanthology.org/2020.tacl-1.35/).


In [28]:
perplexity_df = run_polygraph(Perplexity())
display(perplexity_df)

,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,0.480415,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,0.635060,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


### 🔵 4. Monte Carlo Sequence Entropy

**What it measures:** uncertainty across several alternative responses using their sequence probabilities.

**What it requires:** multiple sampled generations with access to their sequence probabilities.

**How to interpret the score:** higher values indicate greater uncertainty across the sampled generations.

This method is slower than single-generation estimators because it requires multiple additional responses.

For more detail, see [Kuhn et al. (2023)](https://openreview.net/forum?id=VD-AYtP0dve).


In [29]:
mc_entropy_df = run_polygraph(MonteCarloSequenceEntropy())
display(mc_entropy_df)

,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,53.726813,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,99.489028,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


## 🟤 Attention-Based Methods

Attention-based estimators use the model’s attention patterns as an additional internal uncertainty signal.

These methods examine how generated tokens attend to the input and previously generated context. They should not be treated as direct explanations of model reasoning.


### 🟤 1. RAUQ

**What it measures:** uncertainty from recurrent attention patterns observed during generation.

**What it requires:** one model generation from a transformer model that exposes its internal attention tensors.

**How to interpret the score:** higher values generally indicate less stable or less informative attention behaviour and therefore greater uncertainty.

For more detail, see [Vazhentsev et al. (2025)](https://arxiv.org/abs/2502.04964).


In [30]:
rauq_df = run_polygraph(RAUQ())
display(rauq_df)

,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,2.947091,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,3.097114,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


### 🟤 2. Attention Score

**What it measures:** how strongly generated tokens attend back to the input prompt across attention heads and layers.

**What it requires:** one model generation with access to attention weights across the model's layers and attention heads.

**How to interpret the score:** higher uncertainty values indicate that the generated response is less strongly anchored to the input.

For more detail, see [Sriramanan et al. (2024)](https://arxiv.org/abs/2406.10209).


In [31]:
attention_df = run_polygraph(AttentionScore())
display(attention_df)

,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,474.997942,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,802.303743,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


## 🟢 Meaning-Diversity Methods

These methods compare the meanings of multiple generated responses rather than relying only on token probabilities.

LM-Polygraph uses a Natural Language Inference model, typically DeBERTa, to identify whether responses express the same meaning, entail one another, or contradict each other. This auxiliary model does not generate text, and the first run may take longer because it must be downloaded.


### 🟢 1. Semantic Entropy

**What it measures:** uncertainty across semantic clusters formed from multiple sampled responses.

**What it requires:** multiple sampled responses, their sequence probabilities, and a semantic entailment model for grouping responses by meaning.

**How to interpret the score:** higher values indicate that the model produced several competing meanings and is therefore more uncertain.

This method requires multiple sampled generations, so it is slower than single-generation estimators.

For more detail, see [Kuhn et al. (2023)](https://openreview.net/forum?id=VD-AYtP0dve).


In [32]:
semantic_entropy_df = run_polygraph(SemanticEntropy())
display(semantic_entropy_df)

,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,65.550514,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,87.097795,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


### 🟢 2. CoCoA (MSP)

**What it measures:** uncertainty by combining Maximum Sequence Probability with semantic agreement across sampled responses.

**What it requires:** multiple sampled responses, Maximum Sequence Probability scores, and semantic comparison between generations.

**How to interpret the score:** higher values indicate that the primary response is less probable or less consistent with the sampled alternatives, and is therefore more uncertain.

This method requires multiple sampled generations.

For more detail, see [Vashurin et al. (2025)](https://arxiv.org/abs/2502.04964).


In [33]:
cocoa_msp_df = run_polygraph(CocoaMSP())
display(cocoa_msp_df)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

1it [00:01,  1.25s/it]
1it [00:01,  1.82s/it]


,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,12.131277,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,33.541965,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


### 🟢 3. CoCoA (PPL)

**What it measures:** uncertainty by combining perplexity with semantic agreement across sampled responses.

**What it requires:** multiple sampled responses, perplexity scores, and semantic comparison between generations.

**How to interpret the score:** higher values indicate that the response is less probable or less consistent with the sampled alternatives, and is therefore more uncertain.

This method requires multiple sampled generations.

For more detail, see [Vashurin et al. (2025)](https://arxiv.org/abs/2502.04964).


In [34]:
cocoa_ppl_df = run_polygraph(CocoaPPL())
display(cocoa_ppl_df)

1it [00:01,  1.23s/it]
1it [00:01,  1.81s/it]


,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,0.093411,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,0.189611,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


##  🟣 Reflexive Methods

Reflexive estimators ask the model to evaluate the reliability of its own generated response.

### 🟣 1. P(True)

**What it measures:** the model's self-assessed support for its previously generated answer being true.

**What it requires:** the generated response and one additional model evaluation that judges whether the response is true.

**How to interpret the score:** higher LM-Polygraph values indicate that the model assigns less support to its answer being correct and is therefore more uncertain.

This method requires an additional model evaluation step.

For more detail, see [Kadavath et al. (2022)](https://arxiv.org/abs/2207.05221).


In [35]:
p_true_df = run_polygraph(PTrue())
display(p_true_df)

,prompt,uncertainty,response
0,Prompt 1 — Anaphylaxis,7.900523,The most appropriate immediate treatment for a 60-year-old patient with a known penicillin allergy who dev...
1,Prompt 2 — Chest pain,7.344684,The immediate first-line management for a 50-year-old man presenting with crushing chest pain radiating to...


## 🟠 Density-Based Methods

Density-based estimators use the model’s hidden representations to determine how similar a new input is to a fitted reference distribution.

### 🟠 Robust Density Estimation

**What it measures:** how unusual a prompt's hidden representation is relative to a fitted in-domain reference distribution.

**What it requires:** access to model hidden representations and a reference distribution fitted on representative in-domain examples.

**How to interpret the score:** a larger RDE score indicates that the representation is farther from the reference distribution and is therefore considered more uncertain.

For more detail, see [Yoo et al. (2022)](https://aclanthology.org/2022.findings-acl.289/).

---

#### 🔬 How RDE Works Natively

Before using the `uq_toolbox` helper, it helps to understand the two-stage pipeline RDE requires:

**Stage 1 — Fit the reference distribution**
```python
# Load reference questions from an in-domain dataset (e.g. PubMedQA)
# Pass them through the model and extract hidden-state representations
# Fit RDESeq to those representations
estimator = RDESeq(layer="decoder")
manager = UEManager(data=reference_dataset, model=model, estimators=[estimator], ...)
manager()   # fits the reference distribution
```

**Stage 2 — Score new prompts**
```python
# For each new prompt, extract the same hidden-state representation
# Measure how far it lies from the fitted reference distribution
rde_score = manager.estimations[("sequence", "RDESeq")]

```

This two-stage requirement is what makes RDE different from all other estimators in this notebook — it cannot score a prompt without first seeing reference examples.

> **Interpretation:** A larger RDE score indicates that the prompt's hidden representation is more unusual relative to the reference domain, and is therefore considered more uncertain.

For a complete native implementation, see LM-Polygraph's official [Question Answering tutorial](https://github.com/IINemo/lm-polygraph/blob/main/examples/other/qa_example.ipynb).

---

#### 🧰 Implementation via `uq_toolbox`

`run_rde_sequence(...)` wraps the two stages above into a single call. It handles dataset loading, reference fitting, evaluation, and result formatting internally. The method, reference dataset, training size, and representation layer all remain visible in the notebook.

The helper performs the following steps:

1. loads a small reference subset from **PubMedQA**;
2. passes the reference questions through the language model;
3. extracts their decoder hidden-state representations;
4. fits LM-Polygraph's `RDESeq` estimator to those representations;
5. extracts the same representation for each evaluation prompt;
6. measures how unusual each prompt is relative to the fitted reference distribution.

> **Tutorial setting:** only 20 PubMedQA reference questions are used to reduce runtime. This demonstrates the complete RDE workflow but is not sufficient for a fully fitted research setting.

> **Note:** The wrapper does not introduce a new uncertainty method — it only simplifies the native two-stage workflow provided by LM-Polygraph.

In [42]:
import importlib
import uq_toolbox.core.density_uq as density_uq
rde_result = density_uq.run_rde_sequence(
    uq_context=uq_context,
    prompts=prompts,
    prompt_labels=prompt_labels,
    training_size=20,
    layer="decoder",
)

display(pd.DataFrame(rde_result["rows"]).round(4))

Running RDE with 20 reference examples for 2 prompt(s)...


  0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sklearn/covariance/_robust_covariance.py:793: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(
100%|██████████| 2/2 [01:57<00:00, 58.78s/it]

RDE calculation complete.


,Technique,Category,Granularity,Prompt,Uncertainty
0,Robust Density Estimation,Density-based,sequence,Prompt 1 — Anaphylaxis,0.0012
1,Robust Density Estimation,Density-based,sequence,Prompt 2 — Chest pain,0.0011


### 🔎 Interpreting the RDE Output

Each value is a sequence-level representation-distance score.

- A smaller score means that the prompt is more similar to the PubMedQA reference distribution.
- A larger score means that the prompt is more unusual relative to that reference domain.

RDE scores should only be compared across examples produced with the same model, reference dataset, representation layer, and estimator configuration. They should not be interpreted as probabilities of error.

Because only 20 reference questions are used here, the results demonstrate the method rather than provide a fully calibrated uncertainty estimate.

In [44]:
display(pd.DataFrame(rde_result["settings"].items(), columns=["Setting", "Value"]))

,Setting,Value
0,estimator,RDESeq
1,category,Density-based
2,granularity,sequence
3,model_alias,qwen
4,model_name,Qwen/Qwen2.5-0.5B-Instruct
5,reference_dataset,qiaojin/PubMedQA
6,reference_subset,pqa_labeled
7,reference_text_column,question
8,training_size,20
9,batch_size,1


> **Why ensemble methods are excluded:** LM-Polygraph’s sentence- and token-level ensemble measures require multiple trained models or checkpoints. This tutorial uses a single Qwen model, so these high-compute, high-memory methods are not included.

### 📊 LM-Polygraph Summary
Each column represents one uncertainty estimator, while each row represents a medical prompt. This layout makes it easier to compare how the same technique changes across prompts.

Because the estimators are not normalized, values should be compared vertically within the same column, not horizontally across different techniques.

In [43]:
polygraph_frames = {
    "Maximum Sequence Probability": msp_df,
    "Perplexity": perplexity_df,
    "Monte Carlo Sequence Entropy": mc_entropy_df,
    "RAUQ": rauq_df,
    "Attention Score": attention_df,
    "Semantic Entropy": semantic_entropy_df,
    "CoCoA (MSP)": cocoa_msp_df,
    "CoCoA (PPL)": cocoa_ppl_df,
    "P(True)": p_true_df,
}

polygraph_results_df = pd.concat(
    [
        df[["prompt", "uncertainty"]].assign(technique=name)
        for name, df in polygraph_frames.items()
    ],
    ignore_index=True,
)

polygraph_table = polygraph_results_df.pivot(
    index="prompt",
    columns="technique",
    values="uncertainty",
)

display(polygraph_table.round(4))

technique,Attention Score,CoCoA (MSP),CoCoA (PPL),Maximum Sequence Probability,Monte Carlo Sequence Entropy,P(True),Perplexity,RAUQ,Semantic Entropy
prompt,,,,,,,,,
Prompt 1 — Anaphylaxis,474.9979,12.1313,0.0934,58.2335,53.7268,7.9005,0.4804,2.9471,65.5505
Prompt 2 — Chest pain,802.3037,33.5420,0.1896,90.8800,99.4890,7.3447,0.6351,3.0971,87.0978


> **📏Normalization note:** LM-Polygraph does not provide one built-in normalization procedure shared by all estimators. Different normalization approaches can be applied depending on the score distribution and comparison objective. Normalization is discussed separately later in the notebook.

<a id="claim-level"></a>

---

## 🧩 Claim-Level Uncertainty

A generated response may contain several factual statements that are not equally reliable. Sequence-level uncertainty assigns one score to the complete response — claim-level uncertainty instead breaks the response into atomic factual claims and scores each one independently.

This is especially useful in the medical domain, where a response may contain a correct treatment recommendation alongside an uncertain dosage, contraindication, or supporting explanation.

---

#### 🔬 How the Claim-Level Pipeline Works Natively

The pipeline runs five stages in sequence:

| Stage | What happens |
|---|---|
| 1️⃣ **Generate** | Qwen generates a response; token probabilities and logits are collected |
| 2️⃣ **Extract claims** | GPT-4.1-mini splits the response into atomic factual claims via `ClaimsExtractor` |
| 3️⃣ **Prepare statistics** | Entropy, likelihood, and alternative-token statistics are computed |
| 4️⃣ **Semantic comparison** | A DeBERTa NLI model compares each claim with generated alternatives |
| 5️⃣ **Score** | Six estimators assign one uncertainty score per claim |

Natively, the full pipeline looks like this:

```python
from lm_polygraph.stat_calculators import (
    GreedyProbsCalculator, EntropyCalculator,
    GreedyLMProbsCalculator, ClaimsExtractor,
    GreedyAlternativesNLICalculator, ClaimPromptCalculator,
)
from lm_polygraph.estimators import MaximumClaimProbability
from lm_polygraph.utils.deberta import Deberta
from lm_polygraph.utils.openai_chat import OpenAIChat

# Stage 1 — generate and collect token statistics
deps = GreedyProbsCalculator()(deps, texts=[prompt], model=model)

# Stage 2 — extract atomic claims via GPT
deps.update(ClaimsExtractor(OpenAIChat("gpt-4.1-mini"))(deps, texts=[prompt], model=model))

# Stage 3 — prepare entropy and likelihood statistics
deps.update(EntropyCalculator()(deps, texts=[prompt], model=model))
deps.update(GreedyLMProbsCalculator()(deps, texts=[prompt], model=model))

# Stage 4 — semantic comparison with DeBERTa NLI
deps.update(GreedyAlternativesNLICalculator(Deberta())(deps, texts=[prompt], model=model))
deps.update(ClaimPromptCalculator()(deps, texts=[prompt], model=model))

# Stage 5 — score each claim
scores = MaximumClaimProbability()(deps)
```

The six claim-level estimators that can be applied at stage 5 are:

| Estimator | What it measures | What it requires | How to interpret the score |
|---|---|---|---|
| `MaximumClaimProbability` | Uncertainty from the probability of the complete claim | Claim token probabilities from one generated response | Higher values indicate a less probable and therefore more uncertain claim |
| `MaxTokenEntropyClaim` | The largest token-level entropy within the claim | Token-level next-token distributions for the claim | Higher values indicate that at least one claim token was generated with greater uncertainty |
| `PerplexityClaim` | Average token-level surprisal within the claim | Token log-probabilities for all tokens in the claim | Higher values indicate that the claim was less expected by the model |
| `PointwiseMutualInformationClaim` | How strongly the claim depends on its context | Claim likelihood statistics with and without the relevant context | Higher uncertainty values indicate weaker contextual support for the claim |
| `PTrueClaim` | The model's self-assessed support that the claim is true | An extracted claim and an additional truth-evaluation model call | Higher uncertainty values indicate less support for the claim being true |
| `ClaimConditionedProbabilityClaim` | Claim probability conditioned on semantic alternatives | Extracted claims, token statistics, and NLI-based semantic alternatives | Higher values indicate that the claim is less supported when alternatives are considered |

> **Interpretation:** Higher values generally indicate greater uncertainty. Scores use different numerical scales, so compare claims within the same estimator rather than across estimators.

For the complete native API, see the official [LM-Polygraph repository](https://github.com/IINemo/lm-polygraph).

---

#### 🧰 Implementation via `uq_toolbox`

`run_claim_level_uq(...)` packages the full five-stage pipeline above into a single call, handling model generation, OpenAI claim extraction, DeBERTa semantic comparison, calculator execution, and result formatting internally.

```python
claim_result = run_claim_level_uq(
    model=model,
    tokenizer=tokenizer,
    base_model=base_model,
    prompts=prompts,
    prompt_labels=prompt_labels,
    max_new_tokens=50,
)
```

It returns four components:

- `responses` — the locally generated answers;
- `claims` — the extracted atomic claims;
- `dataframe` — one row per prompt, claim, and estimator;
- `settings` — model, claim extractor, NLI model, estimators, and generation configuration.

> **Note:** The wrapper does not introduce a new uncertainty method — it only organises the native LM-Polygraph pipeline and keeps all responses, claims, scores, and settings visible in the notebook.

In [45]:
from uq_toolbox.core.claim_uq import run_claim_level_uq

claim_result = run_claim_level_uq(
    model=model,
    tokenizer=tokenizer,
    base_model=base_model,
    prompts=prompts,
    prompt_labels=prompt_labels,
    max_new_tokens=50,
)

In [46]:
display(
    pd.DataFrame(
        claim_result["settings"].items(),
        columns=["setting", "value"],
    )
)

,setting,value
0,generation_model,Qwen/Qwen2.5-0.5B-Instruct
1,claim_extractor,gpt-4.1-mini
2,claim_extractor_provider,OpenAI
3,nli_model,DeBERTa
4,nli_device,cuda:0
5,granularity,claim
6,estimators,"[Maximum Claim Probability, Max Token Entropy, Perplexity, PMI, p(True), Claim-Conditioned Probability]"
7,max_new_tokens,50


In [47]:
for label, response, claims in zip(
    prompt_labels,
    claim_result["responses"],
    claim_result["claims"],
):
    print(f"\n{label}")
    print("=" * len(label))
    print(f"\nGenerated response:\n{response}")

    print("\nExtracted claims:")
    for index, claim in enumerate(claims, start=1):
        print(f"{index}. {claim.claim_text}")


Prompt 1 — Anaphylaxis

Generated response:
Human skin is the most common site of anaphylactic shock.
A: 
A. Apply a cold compress to the lips and tongue
B: 
B. Administer epinephrine
C: 
C. Administer a 

Extracted claims:
1. Human skin is the most common site of anaphylactic shock.
2. Apply a cold compress to the lips.
3. Apply a cold compress to the tongue.
4. Administer epinephrine.

Prompt 2 — Chest pain

Generated response:
 A: Administer oxygen B: Administer a beta-blocker C: Administer a vasodilator D: Administer a proton pump inhibitor D: Administer a proton pump inhibitor

The immediate first-line management for a patient with crushing chest

Extracted claims:
1. Administer oxygen.
2. Administer a beta-blocker.
3. Administer a vasodilator.
4. Administer a proton pump inhibitor.


In [48]:
claim_scores_df = (
    claim_result["dataframe"]
    .pivot_table(
        index=["prompt", "claim"],
        columns="technique",
        values="uncertainty",
        aggfunc="first",
    )
    .reset_index()
)

display(claim_scores_df.round(4))

technique,prompt,claim,Claim-Conditioned Probability,Max Token Entropy,Maximum Claim Probability,PMI,Perplexity,p(True)
0,Prompt 1 — Anaphylaxis,Administer epinephrine.,-0.2289,3.3689,2.4109,-120.6040,0.4018,9.7431
1,Prompt 1 — Anaphylaxis,Apply a cold compress to the lips.,-0.0483,5.0811,7.2244,-70.7716,1.0321,10.0626
2,Prompt 1 — Anaphylaxis,Apply a cold compress to the tongue.,-0.0786,5.0811,6.6100,-67.6571,0.9443,10.0559
3,Prompt 1 — Anaphylaxis,Human skin is the most common site of anaphylactic shock.,-0.0121,5.3107,14.2287,-131.4202,1.0945,9.6406
4,Prompt 2 — Chest pain,Administer a beta-blocker.,-0.0477,5.6185,6.8906,-82.3144,1.1484,10.1567
5,Prompt 2 — Chest pain,Administer a proton pump inhibitor.,-0.0204,5.6185,7.3421,-65.2078,1.2237,10.2765
6,Prompt 2 — Chest pain,Administer a vasodilator.,-0.0190,5.6185,7.3385,-110.2828,1.0484,11.8747
7,Prompt 2 — Chest pain,Administer oxygen.,-0.1295,5.6185,4.5525,-32.1794,1.5175,10.3575


### 🔎 Interpreting the Claim-Level Output

Each row represents one extracted factual claim, while each estimator column reports that claim’s uncertainty.

- Higher values generally indicate greater uncertainty.
- Scores from different estimators use different scales and should not be compared directly.
- Compare claims within the same estimator to identify which statements are relatively less reliable.



> **Tutorial note:** claim extraction may occasionally include formatting fragments or answer-option labels. These results demonstrate the claim-level workflow rather than a fully calibrated evaluation.

<a id="supervised-uhead"></a>

---

# 4️⃣ Supervised Uncertainty Head

> ⚠️ **Hardware requirement:** This section loads Mistral-7B together with its compatible uncertainty head and therefore requires substantially more GPU memory than the previous sections. A high-memory GPU is recommended.

## 🧠 From Fixed Scores to Learned Uncertainty

Previous methods estimate uncertainty using predefined formulas based on token probabilities, entropy, attention, semantic agreement, or representation distance.

A supervised uncertainty head instead **learns** which internal model patterns are associated with reliable or unreliable generations.

During inference, the head reads the base model's hidden representations and predicts one uncertainty value for each generated token. These token-level values are then aggregated into a single response-level uncertainty score.

This section uses the [LLM Uncertainty Head](https://github.com/IINemo/llm-uncertainty-head) library together with LM-Polygraph's `CausalLMWithUncertainty` generation interface.

The compatible model pair used in this tutorial is:

- `mistralai/Mistral-7B-Instruct-v0.2`
- `llm-uncertainty-head/uhead6_Mistral-7B-Instruct-v0.2`

The uncertainty head must match the base model because it was trained on that model's hidden-representation space.

For further details, see [Shelmanov et al. (2025), *A Head to Predict and a Head to Question*](https://aclanthology.org/2025.emnlp-main.1809/).

---

## 🔬 How `llm-uncertainty-head` Works Natively

Before using the `uq_toolbox` wrapper, it helps to understand what the library does under the hood. The supervised uncertainty head pipeline has three steps:

**1. Load the base model and tokenizer**
```python
tokenizer  = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
base_model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", ...)
```

**2. Load the pretrained uncertainty head and attach it**
```python
uhead         = UncertaintyHead.from_pretrained("llm-uncertainty-head/uhead6_Mistral-7B-Instruct-v0.2")
adapted_model = CausalLMWithUncertainty(model=base_model, uncertainty_head=uhead)
```

**3. Generate — text and token-level uncertainty in one forward pass**
```python
output              = adapted_model.generate(**inputs, max_new_tokens=60)
generated_text      = tokenizer.decode(output.sequences[0])
token_uncertainties = output.uncertainties[0]   # one score per generated token
```

The uncertainty head reads the hidden representations produced during generation and predicts one learned uncertainty score per token. It runs **in parallel with the generation loop** — no second forward pass is needed.

> **Interpretation:** Higher values indicate greater learned uncertainty.

For the complete native API, see the official [LLM Uncertainty Head repository](https://github.com/IINemo/llm-uncertainty-head).

---

## 🧰 Implementation via `uq_toolbox.learned_uq`

`SupervisedUQManager` and `evaluate_supervised_batch` wrap the three steps above into a single reusable interface. Model loading, head attachment, chat-template formatting, and token aggregation are all handled internally, keeping the notebook focused on interpreting results rather than managing infrastructure.

```python
from uq_toolbox.learned_uq import (
    SupervisedUQManager,
    evaluate_supervised_batch,
)
```

### `SupervisedUQManager`

This class:

1. loads the tokenizer and base language model;
2. loads the compatible pretrained uncertainty head;
3. attaches the head to the base model;
4. configures LM-Polygraph's uncertainty-aware generation adapter;
5. enables token-level uncertainty prediction.

### `evaluate_supervised_batch`

This function:

1. applies the model's chat template;
2. tokenises each prompt;
3. generates a response;
4. extracts one learned uncertainty score per generated token;
5. decodes the generated response;
6. aggregates the token-level scores into a single sequence-level score.

The evaluation helper supports **mean**, **maximum**, **median**, and **sum** aggregation. This tutorial uses **mean**.

> **Note:** The wrapper does not introduce a new uncertainty method — it only simplifies the native workflow provided by the original library.

In [49]:
from uq_toolbox.learned_uq import (
    SupervisedUQManager,
    evaluate_supervised_batch,
)

supervised_manager = SupervisedUQManager(
    model_name="mistralai/Mistral-7B-Instruct-v0.2",
    uncertainty_head_name=(
        "llm-uncertainty-head/"
        "uhead6_Mistral-7B-Instruct-v0.2"
    ),
    torch_dtype=torch.float16,
    device_map="auto",
    max_new_tokens=50,
    max_input_length=512,
    do_sample=False,
    attn_implementation="eager",
).load()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/241 [00:00<?, ?B/s]

weights.pth:   0%|          | 0.00/19.7M [00:00<?, ?B/s]

In [53]:
supervised_results = evaluate_supervised_batch(
    prompts=prompts,
    labels=prompt_labels,
    manager=supervised_manager,
    aggregation="mean",
)

supervised_uq_df = pd.DataFrame(supervised_results)

uhead_summary_df = supervised_uq_df[
    [
        "prompt",
        "supervised_uncertainty",
        "aggregation",
        "granularity",
        "estimator_name",
        "model_name",
        "uncertainty_head_name",
        "response",
    ]
].copy()

display(
    uhead_summary_df.round(
        {"supervised_uncertainty": 4}
    )
)

,prompt,supervised_uncertainty,aggregation,granularity,estimator_name,model_name,uncertainty_head_name,response
0,Prompt 1 — Anaphylaxis,0.0853,mean,sequence,learned_uncertainty_head,mistralai/Mistral-7B-Instruct-v0.2,llm-uncertainty-head/uhead6_Mistral-7B-Instruct-v0.2,"The most appropriate immediate treatment for an allergic reaction, specifically anaphylaxis, which is char..."
1,Prompt 2 — Chest pain,0.0333,mean,sequence,learned_uncertainty_head,mistralai/Mistral-7B-Instruct-v0.2,llm-uncertainty-head/uhead6_Mistral-7B-Instruct-v0.2,The immediate first-line management for a 50-year-old man presenting with crushing chest


### 🔬 Token-Level Learned Uncertainty

**What it measures:** learned token-level uncertainty from the language model's internal representations, together with an aggregated sequence-level score.

**What it requires:** a compatible base language model, its matching pretrained uncertainty-head checkpoint, and access to the model representations expected by that head.

**How to interpret the score:** larger token or sequence values indicate greater uncertainty according to the learned head.The supervised uncertainty head assigns one learned uncertainty score to each generated token.

The notebook also reports `supervised_uncertainty`, obtained by aggregating the token-level scores. In this tutorial, the aggregation method is the **mean**.

Token-level inspection is useful because a response may have a moderate average uncertainty while still containing a few highly uncertain tokens.

In [54]:
token_rows = []

for _, row in supervised_uq_df.iterrows():
    for position, (token, score) in enumerate(
        zip(row["generated_tokens"], row["token_uncertainties"]),
        start=1,
    ):
        token_rows.append({
            "prompt": row["prompt"],
            "position": position,
            "token": token,
            "uncertainty": score,
        })

token_scores_df = pd.DataFrame(token_rows)

display(token_scores_df.round({"uncertainty": 4}))

,prompt,position,token,uncertainty
0,Prompt 1 — Anaphylaxis,1,▁The,0.0288
1,Prompt 1 — Anaphylaxis,2,▁most,0.0309
2,Prompt 1 — Anaphylaxis,3,▁appropriate,0.0305
3,Prompt 1 — Anaphylaxis,4,▁immediate,0.0290
4,Prompt 1 — Anaphylaxis,5,▁treatment,0.0200
5,Prompt 1 — Anaphylaxis,6,▁for,0.0395
6,Prompt 1 — Anaphylaxis,7,▁an,0.0283
7,Prompt 1 — Anaphylaxis,8,▁allerg,0.0185
8,Prompt 1 — Anaphylaxis,9,ic,0.1170
9,Prompt 1 — Anaphylaxis,10,▁reaction,0.0378


### 🔎 Interpreting the UHead Scores

- `token_uncertainties` contains one learned uncertainty value for each generated token.
- Larger values indicate greater uncertainty according to the learned head.
- A few highly uncertain tokens may be hidden by the sequence-level average, so both levels should be inspected.
- The scores are specific to the selected Mistral base model, uncertainty-head checkpoint, and aggregation strategy.

The values should therefore be compared only across responses generated with the same model-head configuration.

In [55]:
import plotly.express as px

px.line(
    token_scores_df,
    x="position",
    y="uncertainty",
    color="prompt",
    hover_data=["token"],
    markers=True,
    title="Token-Level Supervised Uncertainty",
).show()

### 📊 Supervised Uncertainty Summary

The uncertainty head assigns a score to each generated token and aggregates those values into one sequence-level uncertainty score.

The graph shows where uncertainty changes across the generated response. Peaks indicate tokens that the uncertainty head considers less reliable, while lower values indicate more confident regions. The sequence-level score summarises this token-level information using the selected aggregation method.

---

# 🧠 Conclusion

This notebook presented a tutorial on **white-box uncertainty quantification for medical-domain large language model responses** using **UQLM**, **LM-Polygraph**, and a **supervised uncertainty head**.

The experiments covered uncertainty at several levels:

* **Token level:** uncertainty associated with individual generated tokens
* **Sequence level:** uncertainty associated with the complete response
* **Claim level:** uncertainty associated with individual factual statements
* **Multi-generation level:** uncertainty estimated from agreement and disagreement across sampled responses

## 🔍 What each approach contributes

With **UQLM**, the notebook demonstrated probability-based, entropy-based, self-evaluation, semantic-consistency, and ensemble methods. These methods used token probabilities, sequence confidence, P(True), and agreement across multiple generations to estimate how strongly the model supported its responses.

With **LM-Polygraph**, the notebook explored a broader range of estimators, including:

* probability-based methods
* entropy-based methods
* semantic and meaning-diversity methods
* attention-based methods
* density-based methods
* reflexive methods

The **claim-level analysis** provided a more fine-grained view by decomposing each medical response into individual factual statements. This is especially important in the medical domain, where a response may appear reliable overall while still containing an uncertain diagnosis, recommendation, or treatment-related claim.

The **supervised uncertainty head** demonstrated how a learned estimator can assign uncertainty at both token and sequence level. Token-level visualisation helps identify the specific regions of a generated response where uncertainty increases, while the aggregated sequence score provides a compact summary of the complete answer.

## ⚠️ Interpretation of the Results

The methods demonstrated in this notebook estimate different forms of uncertainty and may produce scores on different numerical scales. Their raw values should therefore not be treated as directly interchangeable.

A higher score may represent either greater confidence or greater uncertainty depending on the estimator. Score direction must always be checked before interpretation.

Similarly, a score of `0.8` from one method does not necessarily represent the same level of reliability as a score of `0.8` from another method. Meaningful cross-method comparison requires:

* consistent score direction
* normalization
* calibration
* labelled evaluation data
* appropriate benchmarking metrics

Because UQLM and LM-Polygraph may use different compatible model backends, the cross-library results in this notebook are illustrative rather than strictly controlled method-to-method comparisons.

## ✅ Key Takeaway

Uncertainty cannot be represented reliably by one universal score. Different estimators capture different sources of uncertainty, including weak token probabilities, semantic disagreement, unstable hidden representations, and learned uncertainty patterns.

A practical uncertainty-quantification pipeline should therefore combine complementary signals to better understand:

* **when** a model is uncertain
* **where** uncertainty occurs in a response
* **why** different estimators disagree

> 📌 **Scope note:** This notebook covers the **white-box uncertainty quantification** component of the wider group project. Black-box methods, multimodal uncertainty, normalization, calibration, and quantitative benchmarking are covered in the other project components.


---

# 📚 References

## Libraries and General UQ Resources

1. Shelmanov, A., Panov, M., Vashurin, R., Vazhentsev, A., Fadeeva, E., and Baldwin, T. (2025). [*Uncertainty Quantification for Large Language Models*](https://aclanthology.org/2025.acl-tutorials.3/). ACL 2025 Tutorial.

2. Bouchard, D., and Chauhan, M. S. (2025). [*Uncertainty Quantification for Language Models: A Suite of Black-Box, White-Box, LLM Judge, and Ensemble Scorers*](https://openreview.net/forum?id=WOFspd4lq5). Transactions on Machine Learning Research.

3. [UQLM: Uncertainty Quantification for Language Models](https://github.com/cvs-health/uqlm). CVS Health.

4. Fadeeva, E., Vashurin, R., Tsvigun, A., Vazhentsev, A., Petrakov, S., Fedyanin, K., Vasilev, D., Goncharova, E., Panchenko, A., Baldwin, T., Nakov, P., Panov, M., and Shelmanov, A. (2023). [*LM-Polygraph: Uncertainty Estimation for Language Models*](https://aclanthology.org/2023.emnlp-demo.41/). EMNLP 2023 System Demonstrations.

5. Vashurin, R. et al. (2025). [*Benchmarking Uncertainty Quantification Methods for Large Language Models with LM-Polygraph*](https://aclanthology.org/2025.tacl-1.11/). Transactions of the Association for Computational Linguistics.

6. [LM-Polygraph Repository](https://github.com/IINemo/lm-polygraph). IINemo.

7. Manakul, P., Liusie, A., and Gales, M. J. F. (2023). [*SelfCheckGPT: Zero-Resource Black-Box Hallucination Detection for Generative Large Language Models*](https://arxiv.org/abs/2303.08896). EMNLP 2023.

8. Scalena, D., Zotos, L., Fersini, E., Nissim, M., and Üstün, A. (2025). [*EAGer: Entropy-Aware Generation for Adaptive Inference-Time Scaling*](https://arxiv.org/abs/2510.11170).

9. Fomicheva, M., Sun, S., Yankovskaya, L., Blain, F., Guzmán, F., Fishel, M., Aletras, N., Chaudhary, V., and Specia, L. (2020). [*Unsupervised Quality Estimation for Neural Machine Translation*](https://aclanthology.org/2020.tacl-1.35/). Transactions of the Association for Computational Linguistics.



10. Kadavath, S. et al. (2022). [*Language Models (Mostly) Know What They Know*](https://arxiv.org/abs/2207.05221).

11. Kuhn, L., Gal, Y., and Farquhar, S. (2023). [*Semantic Uncertainty: Linguistic Invariances for Uncertainty Estimation in Natural Language Generation*](https://arxiv.org/abs/2302.09664).

12. Vashurin, R. et al. (2025). [*Uncertainty Quantification for LLMs through Minimum Bayes Risk: Bridging Confidence and Consistency*](https://arxiv.org/abs/2502.04964).

13. Farquhar, S., Kossen, J., Kuhn, L., and Gal, Y. (2024). [*Detecting Hallucinations in Large Language Models Using Semantic Entropy*](https://www.nature.com/articles/s41586-024-07421-0). Nature.

14. Kossen, J. et al. (2024). [*Semantic Entropy Probes: Robust and Cheap Hallucination Detection in LLMs*](https://arxiv.org/abs/2406.15927).

15. Qiu, X., and Miikkulainen, R. (2024). [*Semantic Density: Uncertainty Quantification for Large Language Models through Confidence Measurement in Semantic Space*](https://arxiv.org/abs/2405.13845).

16. Chen, J., and Mueller, J. (2023). [*Quantifying Uncertainty in Answers from Any Language Model and Enhancing Their Trustworthiness*](https://arxiv.org/abs/2308.16175).


17. Vazhentsev, A. et al. (2025). [*Uncertainty-Aware Attention Heads: Efficient Unsupervised Uncertainty Quantification for LLMs*](https://arxiv.org/abs/2505.20045).

18. Duan, J., Cheng, H., Wang, S., Zavalny, A., Wang, C., Xu, R., Kailkhura, B., and Xu, K. (2024). [*Shifting Attention to Relevance: Towards the Predictive Uncertainty Quantification of Free-Form Large Language Models*](https://aclanthology.org/2024.acl-long.276/). ACL 2024.

19. Yoo, K. M., Park, D., Kang, J., Lee, S.-W., and Park, W. (2022). [*Detection of Adversarial Examples in Text Classification: Benchmark and Baseline via Robust Density Estimation*](https://aclanthology.org/2022.findings-acl.289/). Findings of ACL 2022.

20. [LM-Polygraph Question Answering and Density-Estimation Tutorial](https://github.com/IINemo/lm-polygraph/blob/main/examples/other/qa_example.ipynb).



21. Shelmanov, A. et al. (2025). [*A Head to Predict and a Head to Question: Pre-trained Uncertainty Quantification Heads for Hallucination Detection in LLM Outputs*](https://aclanthology.org/2025.emnlp-main.1809/). EMNLP 2025.

22. [LLM Uncertainty Head Repository](https://github.com/IINemo/llm-uncertainty-head). IINemo.